In [1]:
import pandas as pd
from pyrfc import Connection

In [2]:
# Параметры подключения к системе SAP HANA
def s4_params():
    return {
        'ashost': 's40ap01.torg.x5.ru',                   # адрес SAP сервера
        'sysnr': '00',                                    # номер системы SAP
        'client': '150',                                  # номер клиента SAP
        'snc_mode': '1',                                  # включение SNC (1 - включено)
        'snc_myname': 'p:CN=SERGEY.AKULICH@X5.RU, C=EN',  # ваше SNC-имя
        'snc_partnername': 'p:CN=SRV.SSO-ABAP-S40@x5.ru',  # SNC-имя SAP сервера
        'lang': 'EN'
    }

def conn_s4():
    try:  
        conn =  Connection(**s4_params())
        return conn
    except Exception as e:
        print("Error in connection:", e)
        return None
    
s4_conn = conn_s4()

In [3]:
import re
import yaml

TEMPLATE = re.compile(r"\{\{\s*([^{}]+)\s*\}\}")

def render_params(obj, context):
    if isinstance(obj, dict):
        return {k: render_params(v, context) for k, v in obj.items()}

    if isinstance(obj, list):
        return [render_params(x, context) for x in obj]

    if isinstance(obj, str):
        def replace(m):
            key = m.group(1)
            return str(context[key])
        return TEMPLATE.sub(replace, obj)
    return obj


def read_config(config_path: str) -> dict:
    with open(config_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    company = cfg["company_config"]
    steps = cfg["steps"]

    rendered_step  = render_params(steps[0], company)
    return rendered_step


In [4]:
import json

cfg = read_config("period_close_config.yaml")

result = s4_conn.call(
    "ZFI_AI_PERIOD_CLOSE_EXEC",
    IV_ACTION_TYPE = cfg["action_type"],
    IV_OBJECT_NAME = cfg["object_name"],
    IV_PARAMS_JSON = json.dumps(cfg["params"], ensure_ascii=False),
    IV_ASYNC = "X" if cfg["async"] else "",
    IV_TEST_RUN = "X" if cfg["test_run"] else ""
)

result

{'ET_MESSAGES': [{'TYPE': 'S',
   'ID': 'ZAI_PERIOD',
   'NUMBER': '001',
   'MESSAGE': 'Job submitted:ZAI_PC_RKO7KO8G_221636 /22163600',
   'LOG_NO': '',
   'LOG_MSG_NO': '000000',
   'MESSAGE_V1': 'Job submitted:ZAI_PC_RKO7KO8G_221636 /22163600',
   'MESSAGE_V2': '',
   'MESSAGE_V3': '',
   'MESSAGE_V4': '',
   'PARAMETER': '',
   'ROW': 0,
   'FIELD': '',
   'SYSTEM': ''}],
 'EV_MESSAGE': 'Job submitted:ZAI_PC_RKO7KO8G_221636 /22163600',
 'EV_RESULT_JSON': '{"status":"submitted","mode":"async","program":"RKO7KO8G","jobname":"ZAI_PC_RKO7KO8G_221636","jobcount":"22163600"}',
 'EV_STATUS': 'A'}

In [5]:
import json

data = json.loads(result['EV_RESULT_JSON'])

result = s4_conn.call('ZFI_AI_PERIOD_CLOSE_EXEC',
    IV_ACTION_TYPE = 'TOOL_READ_JOB_SPOOL',
    IV_OBJECT_NAME = '',
    IV_PARAMS_JSON = json.dumps(
        {"JOBNAME": data["jobname"], "JOBCOUNT": data["jobcount"]}
        # {"JOBNAME": "ZAI_PC_RKO7KO8G_221133", "JOBCOUNT": "22113300"}
    ),
)
result

{'ET_MESSAGES': [{'TYPE': 'S',
   'ID': 'ZAI_PERIOD',
   'NUMBER': '001',
   'MESSAGE': 'Spool read successfully. rqident=7795966. Lines=54',
   'LOG_NO': '',
   'LOG_MSG_NO': '000000',
   'MESSAGE_V1': 'Spool read successfully. rqident=7795966. Lines=54',
   'MESSAGE_V2': '',
   'MESSAGE_V3': '',
   'MESSAGE_V4': '',
   'PARAMETER': '',
   'ROW': 0,
   'FIELD': '',
   'SYSTEM': ''}],
 'EV_MESSAGE': '',
 'EV_RESULT_JSON': '["20.05.2026 22:16:36 Actual settlement: Orders                                                                                1","------------------------------------------------------------------------------------------------------------------------------------","","                                                             Basic list","","Selection","Selection Variant              TEST                           test","Period                         011","Posting Period                 011","Fiscal Year                    2025","Processing Type                1  

In [6]:
import json

def spool_json_to_text(ev_result_json: str, start_from: str | None = None) -> str:
    lines = json.loads(ev_result_json)

    if start_from is not None:
        start_idx = None
        for i, line in enumerate(lines):
            if isinstance(line, str) and start_from in line:
                start_idx = i
                break
        if start_idx is not None:
            lines = lines[start_idx:]

    cleaned_lines = [line for line in lines if isinstance(line, str) and line.strip()]
    return "\n".join(cleaned_lines)

spool_filtered = spool_json_to_text(result['EV_RESULT_JSON'], start_from="Result")
spool_filtered

'Result\nProcessing completed with errors\nCorrect the errors that occurred and\nrepeat the settlement for the senders concerned.\n     Messages | Max. Msg Type |         Error |       Warning |   Information\n            3 |               |             3 |               |\nProcessing Category                         Number\nSettlement Executed\nNo Change\nNot Relevant                                     8\nInappropriate Status                             8\nError                                            3\n-----------------------------------------------------\nObjects Selected                                19\n20.05.2026 22:16:36 Actual settlement: Orders                                                                                1\n------------------------------------------------------------------------------------------------------------------------------------\n                                                              Messages\n--------------------------------------------

In [7]:
from config import build_llm, build_mcp_config
from dotenv import load_dotenv

load_dotenv()

llm = build_llm('OPENROUTER_API_KEY') # return ChatOpenAI format        GROQ_API_KEY   
llm.invoke("Say hello")

AIMessage(content='\n\nHello!😊 How can I assist you today?\n', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 15, 'total_tokens': 95, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 68, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-nano-9b-v2:free', 'system_fingerprint': None, 'id': 'gen-1779304701-cQ19oGGlsTmJKBGhvtHH', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e46d3-1b32-74d2-b603-2e6286d2eb5c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 80, 'total_tokens': 95, 'inpu

In [8]:
def build_spool_prompt(spool_text: str) -> str:
    return f"""
Respond strictly in the specified format:

{{
  "error_count": integer, 
  "errors": [
    {{
      "sender": string|null,
      "msg_type": "E"|"A"|"X",
      "msg_num": string|null,
      "message": string
    }}
  ]
}}

Rules:
- Only messages of the following types (E, A, X) should be considered errors.
- "sender" extruct from a row like "Sender: ...".
- "msg_num" have format 2 letters and 3 numbers, for example, KB101.
- "message" located in column Message text.
- if there are no errors, enter {{"error_count": 0, "errors": []}}

Text:

{spool_text}""".strip()


def analyze_spool(spool_text: str) -> dict:
    prompt = build_spool_prompt(spool_text)
    response = llm.invoke(prompt)
    content = response.content if hasattr(response, "content") else str(response)

    try:
        return json.loads(content)
    except json.JSONDecodeError:
        # если модель вернула лишний текст
        start = content.find("{")
        end = content.rfind("}")
        if start != -1 and end != -1 and end > start:
            return json.loads(content[start:end + 1])
        raise


if __name__ == "__main__":
  
    output = analyze_spool(spool_filtered)
    print(json.dumps(output, ensure_ascii=False, indent=2))

{
  "error_count": 3,
  "errors": [
    {
      "sender": "ORD 5000000 Технический заказ для RU06",
      "msg_type": "E",
      "msg_num": "KD205",
      "message": "Maintain the settlement rule of the sender"
    },
    {
      "sender": "ORD FK01 Заказ производства FK01",
      "msg_type": "E",
      "msg_num": "KD205",
      "message": "Maintain the settlement rule of the sender"
    },
    {
      "sender": "ORD FK02 Заказ производства FK01",
      "msg_type": "E",
      "msg_num": "KD205",
      "message": "Maintain the settlement rule of the sender"
    }
  ]
}
